# 📄 PDF Q&A Study Assistant

A RAG pipeline that extracts text from any PDF and answers 
questions, explains concepts, and generates quizzes from it.

**Stack:** pdfminer · LangChain · ChromaDB ·  via OpenAI API

In [ ]:
!pip install pdfminer.six langchain-core langchain-openai \
    langchain-community langchain-huggingface langchain-text-splitters \
    sentence-transformers chromadb --quiet

## Step 1: Extract text from PDF
pdfminer reads the PDF and returns all text as one plain string.
Update PDF_PATH to point to your file.

In [7]:
from pdfminer.high_level import extract_text

PDF_PATH = "/Users/shruthiraghavan/Downloads/Retrieval-augmented generation - Wikipedia.pdf"

raw_text = extract_text(PDF_PATH)

print(f"✅ Extracted {len(raw_text)} characters")
print("\n--- Preview ---")
print(raw_text[:500])

✅ Extracted 18972 characters

--- Preview ---
Retrieval-augmented generation - Wikipedia

7/5/26, 9:46 PM

Retrieval-augmented generation

Retrieval-augmented  generation  (RAG)  is  a  technique  that  enables  large  language  models
(LLMs)  to  retrieve  and  incorporate  new  information  from  external  data  sources.[1]  With  RAG,
LLMs  first  refer  to  a  specified  set  of  documents,  then  respond  to  user  queries.  These  documents
supplement  information  from  the  LLM's  pre-existing  training  data.[2]  This  allows  LLMs


## Step 2: Convert to LangChain Document
LangChain tools expect a Document object, not a plain string.
We wrap the text in one Document and put it in a list.

In [8]:
from langchain_core.documents import Document

documents = [Document(page_content=raw_text)]

print(f"✅ Created {len(documents)} document")
print(f"Total characters: {len(documents[0].page_content)}")

✅ Created 1 document
Total characters: 18972


## Step 3: Split text into chunks
We split the document into smaller overlapping pieces.
chunk_size=500 — max 500 characters per chunk
chunk_overlap=50 — chunks share 50 characters so context isn't lost at edges

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(documents)

print(f"✅ Split into {len(chunks)} chunks")
print("\n--- Sample chunk ---")
print(chunks[0].page_content)

✅ Split into 56 chunks

--- Sample chunk ---
Retrieval-augmented generation - Wikipedia

7/5/26, 9:46 PM

Retrieval-augmented generation


## Step 4: Create embeddings and store in ChromaDB
Each chunk is converted into a vector (list of numbers) using a 
HuggingFace embedding model. Similar meaning = nearby vectors.
ChromaDB stores these vectors so we can search them later.

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectordb = Chroma.from_documents(chunks, embedding)

print(f"✅ Stored {len(chunks)} chunks in ChromaDB")

✅ Stored 56 chunks in ChromaDB


## Step 5: Set up LLM and build QA chain
We connect GPT-3.5 (via OpenAI API) to ChromaDB.
A custom ask() function ties everything together:
question → fetch top 4 relevant chunks from ChromaDB → pass context to GPT-3.5 → return answer

In [16]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key="your_openai_api_key_here",
    temperature=0
)

retriever = vectordb.as_retriever(search_kwargs={"k": 4})

def ask(question):
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])
    response = llm.invoke(f"Answer based on this context:\n{context}\n\nQuestion: {question}")
    return response.content

print("✅ QA ready!")

✅ QA ready!


## Step 6: Ask questions
Type any question about the PDF — explanations, summaries, or ask it to quiz you!

In [12]:
question = "What is retrieval-augmented generation?"
print(f"Q: {question}")
print(f"\nA: {ask(question)}")

Q: What is retrieval-augmented generation?

A: Retrieval-augmented generation (RAG) is a technique that enhances large language models by incorporating an information-retrieval mechanism. This allows the models to access and utilize additional data beyond their original training set without the need for retraining the model. It improves the quality of document retrieval in vector databases.


In [13]:
# Test 2 - explanation
question = "Explain how RAG works in simple terms"
print(f"Q: {question}")
print(f"\nA: {ask(question)}")

Q: Explain how RAG works in simple terms

A: RAG is a system that helps improve language models by pulling relevant information from databases, documents, or the web before generating responses. This helps the language models stick to the facts and reduce errors. However, sometimes RAG may retrieve misleading sources or struggle to determine which source is accurate when faced with conflicting information.


In [14]:
# Test 2 - explanation
question = "Explain how RAG works in simple terms"
print(f"Q: {question}")
print(f"\nA: {ask(question)}")

Q: Explain how RAG works in simple terms

A: RAG is a system that helps improve language models by pulling relevant information from databases, documents, or the web before generating responses. This helps the language models stick to the facts and reduce errors. However, sometimes RAG may retrieve misleading sources or struggle to determine which source is accurate when faced with conflicting information.


In [15]:
# Test 3 - quiz
question = "Give me 3 quiz questions to test my understanding of RAG"
print(f"Q: {question}")
print(f"\nA: {ask(question)}")

Q: Give me 3 quiz questions to test my understanding of RAG

A: 1. What are the three main components of a Retro block in the RAG model?
2. How does RAG improve the performance of large language models (LLMs)?
3. What is one limitation of the RAG model in terms of model retraining and response reliability?
